# 08 — Word tokenizer comparison

[`07_encoder_bakeoff.ipynb`](07_encoder_bakeoff.ipynb) compared the **subword** tokenizers that
ship with the encoder candidates. This notebook compares the **word** tokenizer feeding the
classical TF-IDF pipeline — a layer nobody had looked at, and which turned out to be broken for
two of the five language tracks.

## The defect

`swiftbench/models.py` built its `word_tfidf` branch on scikit-learn's default
`token_pattern=r"(?u)\b\w\w+\b"`. `\w` matches Unicode *alphanumerics*, which excludes categories
`Mn` and `Mc` — **every Sinhala and Tamil vowel sign**.

Nothing raises. The vectorizer returns tokens, the model fits, the metrics look plausible. The
only symptom is that words differing solely in vowel signs collapse onto one token.

## Three candidates

| tokenizer | what it is |
|---|---|
| `sklearn-default` | `\b\w\w+\b` — the incumbent |
| `regex-unicode` | `[\p{L}\p{M}\p{N}]{2,}` — letters + combining marks + digits |
| `indic-nlp` | `indic_nlp_library`'s `trivial_tokenize`, dispatched on script (Kunchukuttan; the IndicNLPSuite lineage behind IndicBERT, already a candidate in `model-research.md` §4) |

Ranked on two things: **character preservation** (does the tokenizer silently throw text away?)
and **downstream headline metric** on dev.

In [1]:
import sys, warnings, re
from pathlib import Path
warnings.filterwarnings("ignore")

REPO = Path.cwd()
while not (REPO / "ml" / "swiftbench").exists() and REPO != REPO.parent:
    REPO = REPO.parent
sys.path.insert(0, str(REPO / "ml"))

import numpy as np, pandas as pd, regex
from indicnlp.tokenize import indic_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.svm import LinearSVC

import swiftbench as sb
from swiftbench import config, imbalance, metrics, splits, tokenize as sbtok

pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda v: f"{v:.4f}")

LANGS = config.LANGUAGES
SCRIPT = {"sinhala": "si", "tamil": "ta", "english": "en", "singlish": "en", "tamilish": "en"}

sk_tok = re.compile(r"(?u)\b\w\w+\b").findall
rx_tok = regex.compile(r"[\p{L}\p{M}\p{N}]{2,}").findall
def indic_tok(t, lang="en"):
    return [w for w in indic_tokenize.trivial_tokenize(t, lang) if len(w) > 1]

TOKENIZERS = {"sklearn-default": sk_tok, "regex-unicode": rx_tok, "indic-nlp": sbtok.tokenize}
print("split sha:", splits.sha())

split sha: e7b5934392cd


## 1. What each one does to a real ticket

In [2]:
SAMPLES = [
    ("sinhala",  "කවුරු හරි මගේ කාඩ් එක පාවිච්චි කරලා"),
    ("sinhala",  "මගේ කාඩ් එක ට්‍රැක් කරලා දෙන්න පුළුවන්ද?"),   # contains ZWJ conjunct
    ("tamil",    "இது மற்றும் கவலை உதவி பாவிச்சி"),
    ("singlish", "mage card eka track karanna puluwanda"),
    ("english",  "someone has used my card"),
]
for lang, text in SAMPLES:
    print(f"[{lang}] {text}")
    for name, fn in TOKENIZERS.items():
        print(f"   {name:16s} {fn(text)}")
    print()

[sinhala] කවුරු හරි මගේ කාඩ් එක පාවිච්චි කරලා
   sklearn-default  ['කව', 'හර', 'මග', 'එක', 'කරල']
   regex-unicode    ['කවුරු', 'හරි', 'මගේ', 'කාඩ්', 'එක', 'පාවිච්චි', 'කරලා']
   indic-nlp        ['කවුරු', 'හරි', 'මගේ', 'කාඩ්', 'එක', 'පාවිච්චි', 'කරලා']

[sinhala] මගේ කාඩ් එක ට්‍රැක් කරලා දෙන්න පුළුවන්ද?
   sklearn-default  ['මග', 'එක', 'කරල', 'වන']
   regex-unicode    ['මගේ', 'කාඩ්', 'එක', 'ට්', 'රැක්', 'කරලා', 'දෙන්න', 'පුළුවන්ද']
   indic-nlp        ['මගේ', 'කාඩ්', 'එක', 'ට්\u200dරැක්', 'කරලා', 'දෙන්න', 'පුළුවන්ද']

[tamil] இது மற்றும் கவலை உதவி பாவிச்சி
   sklearn-default  ['இத', 'மற', 'கவல', 'உதவ']
   regex-unicode    ['இது', 'மற்றும்', 'கவலை', 'உதவி', 'பாவிச்சி']
   indic-nlp        ['இது', 'மற்றும்', 'கவலை', 'உதவி', 'பாவிச்சி']

[singlish] mage card eka track karanna puluwanda
   sklearn-default  ['mage', 'card', 'eka', 'track', 'karanna', 'puluwanda']
   regex-unicode    ['mage', 'card', 'eka', 'track', 'karanna', 'puluwanda']
   indic-nlp        ['mage', 'card', 'eka', 'track'

## 2. Character preservation

The measurement that makes the defect undeniable: what fraction of non-whitespace characters
survive tokenization. A correct tokenizer keeps ~100% (it only drops punctuation and spaces).

In [3]:
dev = splits.get(LANGS, "dev")
rows = []
for lang in LANGS:
    texts = dev[dev.language == lang][config.TEXT_COLUMN].tolist()
    total = sum(len(re.sub(r"\s", "", t)) for t in texts)
    row = {"language": lang, "rows": len(texts)}
    for name, fn in TOKENIZERS.items():
        kept = sum(sum(len(w) for w in fn(t)) for t in texts)
        row[name] = kept / total
    rows.append(row)

preservation = pd.DataFrame(rows)
display(preservation.style.format({k: "{:.1%}" for k in TOKENIZERS}))
worst = preservation["sklearn-default"].min()
print(f"sklearn-default retains as little as {worst:.1%} of characters on one track.")

,language,rows,sklearn-default,regex-unicode,indic-nlp
0,english,1498,93.6%,93.6%,93.6%
1,sinhala,1498,59.9%,96.8%,97.0%
2,singlish,1498,97.4%,97.4%,97.4%
3,tamil,1498,30.7%,97.4%,97.5%
4,tamilish,1498,96.3%,96.3%,96.3%


sklearn-default retains as little as 30.7% of characters on one track.


## 3. Where `regex-unicode` still gets Sinhala wrong

`regex` recovers the vowel signs but excludes **ZWJ (U+200D)** — Unicode category `Cf` — which
Sinhala uses to form conjunct consonants. `ට්‍රැක්` ("track") splits into `ට්` + `රැක්`.

`research/README.md` §3.19.3 flags the ZWJ gotcha as recurring across Sinhala tooling. It is not
an edge case here: "track my card" is a common banking request.

In [4]:
agree = []
for lang in LANGS:
    texts = dev[dev.language == lang][config.TEXT_COLUMN].tolist()
    same = sum(rx_tok(t) == sbtok.tokenize(t) for t in texts)
    zwj = sum("‍" in t for t in texts)
    agree.append({"language": lang, "regex == indic-nlp": same / len(texts),
                  "rows containing ZWJ": zwj / len(texts)})
display(pd.DataFrame(agree).style.format({"regex == indic-nlp": "{:.1%}",
                                          "rows containing ZWJ": "{:.1%}"}))

ex = [t for t in dev[dev.language == "sinhala"][config.TEXT_COLUMN] if "‍" in t][:1]
for t in ex:
    print("text        :", t)
    print("regex       :", rx_tok(t))
    print("indic-nlp   :", sbtok.tokenize(t))

,language,regex == indic-nlp,rows containing ZWJ
0,english,100.0%,0.0%
1,sinhala,92.4%,7.1%
2,singlish,100.0%,0.0%
3,tamil,99.5%,0.0%
4,tamilish,100.0%,0.0%


text        : මගේ කාඩ් එක ට්‍රැක් කරලා දෙන්න පුළුවන්ද?
regex       : ['මගේ', 'කාඩ්', 'එක', 'ට්', 'රැක්', 'කරලා', 'දෙන්න', 'පුළුවන්ද']
indic-nlp   : ['මගේ', 'කාඩ්', 'එක', 'ට්\u200dරැක්', 'කරලා', 'දෙන්න', 'පුළුවන්ද']


## 4. Does it change the model?

Character preservation is the correctness argument. This is the consequence argument: refit the
classical champion under each tokenizer, on the same split, and score dev.

Two feature configurations, because they answer different questions:

- **word only** — isolates the tokenizer's contribution.
- **word + char_wb** — the production config. `char_wb` reads raw characters and is immune to the
  defect, so it may well be masking it.

In [5]:
def build(tok_fn, word_only=False):
    word = TfidfVectorizer(analyzer="word", ngram_range=(1, 2), min_df=2, max_df=0.98,
                           sublinear_tf=True, max_features=25_000,
                           tokenizer=tok_fn, token_pattern=None)
    if word_only:
        return word
    char = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=2,
                           sublinear_tf=True, max_features=50_000)
    return FeatureUnion([("word_tfidf", word), ("char_tfidf", char)])


def score_config(task, langs, tok_fn, word_only):
    col = sb.data.label_column(task)
    tr = imbalance.resample(splits.get(langs, "train"), col, "class_weight")
    dv = splits.get(langs, "dev")
    C = 0.5 if task == "sentiment" else 1.0
    pipe = Pipeline([("f", build(tok_fn, word_only)),
                     ("c", LinearSVC(C=C, class_weight="balanced", dual="auto",
                                     max_iter=5000, random_state=config.RANDOM_STATE))])
    pipe.fit(tr[config.TEXT_COLUMN], tr[col])
    return metrics.score(dv[col], pipe.predict(dv[config.TEXT_COLUMN]), task)["headline"]


rows = []
for task in ["intent", "priority", "sentiment"]:
    for lang in ["sinhala", "tamil", "english"]:
        for word_only in [True, False]:
            r = {"task": task, "language": lang,
                 "features": "word only" if word_only else "word + char_wb"}
            for name, fn in TOKENIZERS.items():
                r[name] = score_config(task, [lang], fn, word_only)
            rows.append(r)
            print(f"  {task:9s} {lang:8s} {r['features']:15s} "
                  + "  ".join(f"{k} {r[k]:.4f}" for k in TOKENIZERS), flush=True)

impact = pd.DataFrame(rows)
impact["indic vs sklearn"] = impact["indic-nlp"] - impact["sklearn-default"]
display(impact)

  intent    sinhala  word only       sklearn-default 0.9008  regex-unicode 0.9189  indic-nlp 0.9170


  intent    sinhala  word + char_wb  sklearn-default 0.9253  regex-unicode 0.9265  indic-nlp 0.9266


  intent    tamil    word only       sklearn-default 0.8100  regex-unicode 0.8652  indic-nlp 0.8658


  intent    tamil    word + char_wb  sklearn-default 0.9155  regex-unicode 0.9157  indic-nlp 0.9157


  intent    english  word only       sklearn-default 0.8812  regex-unicode 0.8812  indic-nlp 0.8812


...

  sentiment tamil    word only       sklearn-default 0.5137  regex-unicode 0.5250  indic-nlp 0.5250


  sentiment tamil    word + char_wb  sklearn-default 0.6013  regex-unicode 0.6250  indic-nlp 0.6250


  sentiment english  word only       sklearn-default 0.5478  regex-unicode 0.5478  indic-nlp 0.5478


  sentiment english  word + char_wb  sklearn-default 0.5921  regex-unicode 0.5921  indic-nlp 0.5921


,task,language,features,sklearn-default,regex-unicode,indic-nlp,indic vs sklearn
0,intent,sinhala,word only,0.9008,0.9189,0.9170,0.0162
1,intent,sinhala,word + char_wb,0.9253,0.9265,0.9266,0.0012
2,intent,tamil,word only,0.8100,0.8652,0.8658,0.0558
3,intent,tamil,word + char_wb,0.9155,0.9157,0.9157,0.0002
4,intent,english,word only,0.8812,0.8812,0.8812,0.0000
5,intent,english,word + char_wb,0.9031,0.9031,0.9031,0.0000
6,priority,sinhala,word only,0.8984,0.9065,0.9060,0.0075
7,priority,sinhala,word + char_wb,0.9079,0.9088,0.9088,0.0009
8,priority,tamil,word only,0.8523,0.8889,0.8889,0.0366
9,priority,tamil,word + char_wb,0.9109,0.9028,0.9028,-0.0081


In [6]:
print("Mean gain of indic-nlp over sklearn-default, by feature configuration:")
display(impact.groupby("features")["indic vs sklearn"].agg(["mean", "min", "max"]))
print("\nBy language (word-only features -- the tokenizer's own contribution):")
display(impact[impact.features == "word only"]
        .pivot_table(index="language", values="indic vs sklearn", aggfunc="mean"))

impact.to_csv(REPO / "ml" / "reports" / "word_tokenizer_comparison.csv", index=False)
preservation.to_csv(REPO / "ml" / "reports" / "word_tokenizer_preservation.csv", index=False)
print("\nwrote word_tokenizer_comparison.csv, word_tokenizer_preservation.csv")

Mean gain of indic-nlp over sklearn-default, by feature configuration:


,mean,min,max
features,,,
word + char_wb,0.0020,-0.0081,0.0237
word only,0.0200,0.0000,0.0558



By language (word-only features -- the tokenizer's own contribution):


,indic vs sklearn
language,
english,0.0000
sinhala,0.0254
tamil,0.0346



wrote word_tokenizer_comparison.csv, word_tokenizer_preservation.csv


## 6. Romanized tracks — is word tokenization even the problem?

Sections 2-4 are about Indic *script*. Singlish and Tamilish are Latin script, where segmentation
is easy: sklearn, regex and indic-nlp all score **exactly 0.0000 apart** on those tracks. So the
romanized question is a different one, and worth answering on its own terms.

The research warns about two distinct romanized failures:

1. **Spelling variance** — `kaard`/`card`/`kad`, `akount`/`akkount`. No standard orthography, so
   one concept fragments across many token types and dev sees words train never did.
2. **Subword shattering** — Script Sensitivity's 312x perplexity finding: genuinely Sinhala
   romanized words break into meaningless pieces, while romanized words that look English
   tokenize cleanly.

The first is measurable directly, as **dev OOV rate**: the share of dev word tokens never seen in
train. High OOV is spelling variance destroying generalization.

In [7]:
from collections import Counter

train_all, dev_all = splits.get(LANGS, "train"), splits.get(LANGS, "dev")
rows = []
for lang in LANGS:
    a = [w for t in train_all[train_all.language == lang][config.TEXT_COLUMN] for w in sbtok.tokenize(t)]
    b = [w for t in dev_all[dev_all.language == lang][config.TEXT_COLUMN] for w in sbtok.tokenize(t)]
    vocab, counts = set(a), Counter(a)
    rows.append({"language": lang, "train types": len(vocab),
                 "dev OOV": sum(w not in vocab for w in b) / len(b),
                 "hapax share": sum(1 for w, n in counts.items() if n == 1) / len(vocab)})
oov = pd.DataFrame(rows)
display(oov.style.format({"dev OOV": "{:.2%}", "hapax share": "{:.1%}"}))

,language,train types,dev OOV,hapax share
0,english,2467,1.18%,38.4%
1,sinhala,2756,1.28%,38.6%
2,singlish,2638,1.18%,37.9%
3,tamil,6197,3.71%,46.1%
4,tamilish,5856,3.41%,47.7%


**Singlish OOV is 1.18% — identical to English.** There is no spelling-variance problem in our
Singlish at all.

That is not good news, it is a measurement of our data. Singlish here is *rule-generated* from
Sinhala by `singlishify.py` with a fixed override table (`කාඩ්` always becomes `card`, never
`kaard`), so the orthography is perfectly consistent by construction. `model-research.md` §5 calls
the romanized cells "an optimistic upper bound"; this is that caveat with a number attached.

Tamil and Tamilish sit at ~3.5% OOV with more than twice the vocabulary (5,856-6,197 types against
~2,600), because they came from the translation pass rather than a deterministic rule — which is
also the likeliest driver of Tamilish being the weakest track everywhere else in the project.

**Conclusion: our data cannot answer the romanized-tokenization question.** Real human-typed
Singlish would show the variance; ours cannot. Any romanized tokenizer would look good here.

In [8]:
# The purpose-built alternative, measured rather than assumed:
# deshanksuman/romanized-sinhala-tokenizer, from the Swa-Bhasha lineage in research/.
from transformers import AutoTokenizer

rom = AutoTokenizer.from_pretrained("deshanksuman/romanized-sinhala-tokenizer")
rows = []
for lang in ["singlish", "tamilish"]:
    s = dev_all[dev_all.language == lang][config.TEXT_COLUMN].head(400).tolist()
    n_words = sum(len(t.split()) for t in s)
    rows.append({"language": lang,
                 "swa-bhasha fertility": sum(len(rom.tokenize(t)) for t in s) / n_words,
                 "our word tokenizer": sum(len(sbtok.tokenize(t)) for t in s) / n_words})
display(pd.DataFrame(rows))

sample = dev_all[dev_all.language == "singlish"][config.TEXT_COLUMN].iloc[0]
print("text            :", sample)
print("swa-bhasha      :", rom.tokenize(sample)[:16])
print("our tokenizer   :", sbtok.tokenize(sample)[:16])

,language,swa-bhasha fertility,our word tokenizer
0,singlish,2.6359,1.0083
1,tamilish,3.0788,1.0406


text            : mage alut card eka evuve kavadada?
swa-bhasha      : ['mage', 'Ġ', 'al', 'ut', 'Ġ', 'c', 'ard', 'Ġ', 'eka', 'Ġ', 'ev', 'uv', 'e', 'Ġ', 'ka', 'vad']
our tokenizer   : ['mage', 'alut', 'card', 'eka', 'evuve', 'kavadada']


**The purpose-built romanized tokenizer is rejected on measurement.** It fragments English
loanwords (`card` becomes `c` + `ard`) and emits its space marker `Ġ` as standalone tokens,
landing at 2.64-3.08 tokens/word — worse than XLM-R's 2.23/2.35 and far worse than plain word
tokenization. It was evidently fitted to a different romanization convention than
`singlish_overrides.py` produces.

**What actually carries romanized meaning here is `char_wb`,** already in the production feature
union. `models.py` says so in its own docstring — `card eka` / `kaard eka` / `card-ai` share signal
only at the character level — and section 4 shows why it matters: `char_wb` is what closes the gap
the broken word tokenizer opened.

So for romanized text the answer is: **keep `char_wb`, keep plain word tokenization, and do not
adopt a specialised romanized tokenizer on this data** — we could not detect a benefit even if one
existed. Revisit when there is human-typed romanized text, alongside Strategy A in
[`21_technique_strategy_a_transliteration.ipynb`](21_technique_strategy_a_transliteration.ipynb).

## 7. Verdict

**`indic-nlp` is adopted** as the word tokenizer in `swiftbench.tokenize`, wired into
`models.py`. It is the only one of the three that preserves 100% of characters on every track
*and* keeps ZWJ conjuncts intact, and it is the research-standard tool for these scripts rather
than a regex written here.

Read the two feature configurations separately, because they say different things:

- On **word-only** features the defect is large — this is the honest size of the bug.
- On **word + char_wb** the gain shrinks toward zero, because `char_wb` was quietly compensating.
  That is why the defect survived this long: the production metric barely moved.

The second observation is not a reason to leave it broken. The mangled tokens were feeding
everything built on word features — including the mined lexicon in
[`20_technique_lexicon_correction.ipynb`](20_technique_lexicon_correction.ipynb), which was
extracting `කව හර` instead of `කවුරු හරි` — and any future word-level analysis would have
inherited it silently.

**Not changed:** `char_wb` (immune by construction) and the encoders' own subword tokenizers,
which are the subject of notebook 07.